# Tugas Praktikum JS06 - Dataset PBL
## Pencarian Kemiripan Berita Hoaks Menggunakan ENN dan ANN (Annoy & HNSW)


### 1. Load dataset PBL


In [1]:
import pandas as pd
import numpy as np
import time

df = pd.read_csv('data/pbl_hoaks_valid_clean.csv')
print('Dimensi data:', df.shape)
df.head()


Dimensi data: (1498, 2)


,teks_bersih,label
0,presiden prabowo minta maaf ikut bop akan kemb...,1
1,video presiden prabowo akan membagikan bantuan...,1
2,pemerintah kota tangerang selatan akan legalka...,1
3,tautan rekrutmen cpns kementerian imigrasi,1
4,ramuan rebusan jahe kunyit kencur dan serai be...,1


### 2. Ekstraksi vektor fitur TF-IDF


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500)
X = tfidf.fit_transform(df['teks_bersih']).toarray().astype('float32')
dim = X.shape[1]
print(f'Dimensi vektor fitur: {X.shape[0]} baris x {dim} fitur')


Dimensi vektor fitur: 1498 baris x 500 fitur


### 3. Exact Nearest Neighbors (Scikit-Learn Brute-force Cosine)


In [3]:
from sklearn.neighbors import NearestNeighbors

t0 = time.time()
nn_exact = NearestNeighbors(n_neighbors=5, metric='cosine', algorithm='brute')
nn_exact.fit(X)
build_time_exact = time.time() - t0

# Query benchmark untuk 100 sampel pertama
query_data = X[:100]
t0 = time.time()
exact_distances, exact_indices = nn_exact.kneighbors(query_data)
query_time_exact = (time.time() - t0) / 100

print(f'Build Time Exact: {build_time_exact:.6f} detik')
print(f'Rata-rata Query Time Exact: {query_time_exact:.6f} detik per kueri')


Build Time Exact: 0.001117 detik
Rata-rata Query Time Exact: 0.000365 detik per kueri


### 4. Approximate Nearest Neighbors - Annoy


In [4]:
from annoy import AnnoyIndex

t0 = time.time()
annoy_index = AnnoyIndex(dim, 'angular')
for i in range(len(X)):
    annoy_index.add_item(i, X[i])
annoy_index.build(10)
build_time_annoy = time.time() - t0

t0 = time.time()
annoy_indices = [annoy_index.get_nns_by_vector(query_data[i], 5) for i in range(100)]
query_time_annoy = (time.time() - t0) / 100

print(f'Build Time Annoy: {build_time_annoy:.6f} detik')
print(f'Rata-rata Query Time Annoy: {query_time_annoy:.6f} detik per kueri')


Build Time Annoy: 0.115279 detik
Rata-rata Query Time Annoy: 0.000322 detik per kueri


### 5. Approximate Nearest Neighbors - HNSW


In [5]:
import hnswlib

t0 = time.time()
p_hnsw = hnswlib.Index(space='cosine', dim=dim)
p_hnsw.init_index(max_elements=len(X), ef_construction=100, M=16)
p_hnsw.add_items(X, np.arange(len(X)))
p_hnsw.set_ef(50)
build_time_hnsw = time.time() - t0

t0 = time.time()
hnsw_labels, hnsw_distances = p_hnsw.knn_query(query_data, k=5)
query_time_hnsw = (time.time() - t0) / 100

print(f'Build Time HNSW: {build_time_hnsw:.6f} detik')
print(f'Rata-rata Query Time HNSW: {query_time_hnsw:.6f} detik per kueri')


Build Time HNSW: 0.029887 detik
Rata-rata Query Time HNSW: 0.000036 detik per kueri


### 6. Tabel Perbandingan Kinerja ENN vs ANN


In [6]:
# Hitung overlap / recall k=5 terhadap exact
overlap_annoy = np.mean([len(set(exact_indices[i]).intersection(set(annoy_indices[i]))) / 5 for i in range(100)])
overlap_hnsw = np.mean([len(set(exact_indices[i]).intersection(set(hnsw_labels[i]))) / 5 for i in range(100)])

df_komparasi = pd.DataFrame({
    'Metode': ['Exact NN (Brute)', 'Annoy', 'HNSW'],
    'Build Time (detik)': [build_time_exact, build_time_annoy, build_time_hnsw],
    'Query Time (detik)': [query_time_exact, query_time_annoy, query_time_hnsw],
    'Recall / Overlap': [1.0, overlap_annoy, overlap_hnsw]
})
print(df_komparasi)


             Metode  Build Time (detik)  Query Time (detik)  Recall / Overlap
0  Exact NN (Brute)            0.001117            0.000365             1.000
1             Annoy            0.115279            0.000322             0.850
2              HNSW            0.029887            0.000036             0.994


### 7. Simulasi Pencarian Kemiripan Berita Baru


In [7]:
sample_idx = 0
query_text = df['teks_bersih'].iloc[sample_idx]
query_vec = X[sample_idx:sample_idx+1]

print('Kueri Berita:')
print(query_text[:120], '...')
print('Label Asli:', 'Hoaks' if df['label'].iloc[sample_idx] == 1 else 'Valid')

neighbors, dists = p_hnsw.knn_query(query_vec, k=3)
print('Top 3 Berita Terdekat (HNSW):')
for rank, (idx, dist) in enumerate(zip(neighbors[0], dists[0]), 1):
    lbl = 'Hoaks' if df['label'].iloc[idx] == 1 else 'Valid'
    print(f"{rank}. [Jarak: {dist:.4f}] [{lbl}] {df['teks_bersih'].iloc[idx][:100]}...")


Kueri Berita:
presiden prabowo minta maaf ikut bop akan kembalikan rp triliun menteri luar negeri ri sugiono mengatakan bahwa iuran bo ...
Label Asli: Hoaks
Top 3 Berita Terdekat (HNSW):
1. [Jarak: 0.0000] [Hoaks] presiden prabowo minta maaf ikut bop akan kembalikan rp triliun menteri luar negeri ri sugiono menga...
2. [Jarak: 0.5540] [Hoaks] bobibos tipu kdm rp triliun...
3. [Jarak: 0.5918] [Hoaks] presiden prabowo bebaskan ammar zoni dari lapas nusakambangan...
